In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
  repo_owner="DataTalksClub",
  repo_name="llm-zoomcamp",
  commit_id="8c1834d",
  allowed_extensions={"md"},
  filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
documents = []

for file in files:
  doc = file.parse()
  documents.append(doc)

# Q1. How many lesson pages
A: 72

In [4]:
len(documents)

72

# Q2. Indexing and searching

In [5]:
from minsearch import Index

index = Index(
  text_fields=['content'],
  keyword_fields=['filename']
)

index.fit(documents)

In [6]:
search_results = index.search(
    query="How does the agentic loop keep calling the model until it stops?",
)

## What's the filename of the first result?
A: `01-agentic-rag/lessons/14-agentic-loop.md`

# Q3. RAG
Q: How many input (prompt) tokens did we send to the model for this request?  
A: 11894

In [8]:
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()
anthropic_client = Anthropic()

In [24]:
from rag_helper import RAGBase

assistant = RAGBase(
  index=index,
  llm_client=anthropic_client
)

In [12]:
response = assistant.rag("How does the agentic loop keep calling the model until it stops?")

In [17]:
print(response['input_tokens'])

11894


# Q4. Chunking
Q: How many chunks do you get?  
A: 295

In [18]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [19]:
len(chunks)

295

# Q5. RAG with chunking
Q: Compare the input tokens with Q3. How many fewer input tokens does the chunked version send?  
A: 2x fewer

In [21]:
from minsearch import Index

chunk_index = Index(
  text_fields=['content'],
  keyword_fields=['filename']
)

chunk_index.fit(chunks)

In [23]:
from rag_helper import RAGBase

chunk_assistant = RAGBase(
  index=chunk_index,
  llm_client=anthropic_client
)

In [29]:
chunk_response = chunk_assistant.rag("How does the agentic loop keep calling the model until it stops?")

In [30]:
print(chunk_response['message'])

# How the Agentic Loop Keeps Calling the Model Until It Stops

The agentic loop uses a **`while True` loop with a flag-based exit condition** to keep calling the model until it stops requesting function calls.

## The Core Mechanism

```python
while True:
    # Call the model
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )
    
    # Check if there are function calls in the response
    has_function_calls = False
    
    for item in response.output:
        if item.type == "function_call":
            # Execute the function call
            has_function_calls = True
        elif item.type == "message":
            # Model returned a message
            pass
    
    # Exit condition: if no function calls this turn, we're done
    if has_function_calls == False:
        break
```

## How It Works

1. **Call the model** with the accumulated message history
2. **Check the response** for function calls

In [31]:
chunk_response['input_tokens']

5531

# Q6. Turning it into an agent
Q: How many times did the agent call `search`?  
A: 4

In [58]:
from toyaikit.llm import AnthropicClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import AnthropicMessagesRunner, DisplayingRunnerCallback

In [45]:
def search(query: str) -> dict[str, str]:
    """
    Search the content database for entries matching the given query.
    """

    return chunk_index.search(
        query        
    ) # type: ignore

In [46]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [47]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the content database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [50]:
instructions = """
You're a course teaching assistant. 
Answer the student's question using the search tool. Make multiple searches with different keywords before answering.
""".strip()

In [59]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = AnthropicMessagesRunner(
  tools=agent_tools,
  developer_prompt=instructions,
  chat_interface=chat_interface,
  llm_client=AnthropicClient(model="claude-haiku-4-5")
)

In [60]:
result = runner.loop(
  prompt="How does the agentic loop work, and how is it different from plain RAG?",
  callback=callback,
)

-> Response received


-> Response received


Aspect,Plain RAG,Agentic Loop
Flow,Fixed sequence,"Dynamic, LLM decides"
Search count,Always 1,0 or more (LLM decides)
Error recovery,None - fails silently,Can retry with different terms
Flexibility,Rigid,Adaptive
Cost,Low (1 API call),Higher (multiple API calls possible)
Latency,Fast,Slower (more round-trips)
